# Glaucoma and cataract classifiers (ODIR-5K): train, evaluate, export for the app

**What this does.** Trains one binary classifier per condition (glaucoma, cataract) with the same recipe as the
DR model, splits **by patient** (no person appears in more than one split), picks each decision threshold on the
validation split only, measures the held-out test split once, and exports `model.tflite` + `model_card.json`
in exactly the format the app's `ModelRegistry` loads. Every number in the model cards is computed here.

**Binary only.** ODIR-5K has diagnostic keywords per eye but **no severity grades**, so these models output
condition present / absent. No 1-5 severity band is produced (the app shows "binary result only").

### Setup (once)
1. Kaggle: **Create -> New Notebook -> File -> Import Notebook** and choose this file.
2. Right panel: **Accelerator: GPU T4 x2**, **Internet: On** (downloads the ImageNet weights).
3. **Add Input** -> search `ocular-disease-recognition-odir5k` (by *andrewmvd*) -> **Add**.
4. **Run All**. Expected time: 45-90 minutes (about 10 minutes to load images, then training).
5. At the end, download **`odir_models.zip`** from the Output panel and send it to Claude Code (it unzips into the
   repo's `models/` folder). Screenshot the final **CHECKS** cell.

The last cell prints PASS/FAIL integrity checks and a plain verdict on whether the results can be trusted.
Weak results are reported as they are; the test split is never used to tune anything.

## 1. Settings

In [ ]:
import glob, hashlib, json, os, re, time, zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)            # the app's input size (models/dr/model_card.json)
TARGET_SENSITIVITY = 0.85        # proposal targets, same as DR
TARGET_SPECIFICITY = 0.80
MIN_TEST_POSITIVES = 30          # fewer positive test eyes than this: metrics too unstable to trust
MIN_TRAIN_POSITIVES = 150        # fewer positive training eyes than this: model unlikely to learn the condition
BOOTSTRAP = 1000

# Smoke-test mode runs the code path on a tiny fake dataset (used by the developer before spending GPU time).
SMOKE = os.environ.get("ODIR_SMOKE_TEST") == "1"
INPUT_ROOT = os.environ.get("ODIR_INPUT_ROOT", "/kaggle/input")
OUT_ROOT = os.environ.get("ODIR_OUT_ROOT", "/kaggle/working")
EPOCHS_HEAD = 2 if SMOKE else 12
EPOCHS_FINE = 1 if SMOKE else 6
BATCH = 8 if SMOKE else 32

CONDITIONS = {
    "glaucoma": {"display_name": "Glaucoma", "positive": "glaucoma", "ambiguous": ["suspected glaucoma"],
                 "extra_exclusions": ["optic disk photographically invisible"]},
    "cataract": {"display_name": "Cataract", "positive": "cataract", "ambiguous": ["suspected cataract"],
                 "extra_exclusions": []},
}
# Keywords meaning the photo itself is unusable for any judgement.
UNUSABLE = ["low image quality", "image offset", "no fundus image", "anterior segment image"]

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"), "| smoke test:", SMOKE)

## 2. Find the dataset

In [ ]:
def find_first(names, want_dir=False, max_depth=6):
    """First match for any of ``names`` under INPUT_ROOT, preferring the earlier name and then the shallowest path.
    Searching instead of assuming one layout works for both Kaggle mount styles:
    /kaggle/input/<dataset>/... and /kaggle/input/datasets/<owner>/<dataset>/..."""
    best = None
    for root, dirs, files in os.walk(INPUT_ROOT):
        depth = root[len(INPUT_ROOT):].count(os.sep)
        if depth >= max_depth:
            dirs[:] = []
        for entry in (dirs if want_dir else files):
            if entry in names:
                rank = (names.index(entry), depth, os.path.join(root, entry))
                best = min(best, rank) if best else rank
        dirs[:] = [d for d in dirs if d not in ("preprocessed_images", "Testing Images")]  # never needed; faster
    return best[2] if best else None

ANNOTATIONS = find_first(["data.xlsx", "full_df.csv"])
IMAGE_DIR = find_first(["Training Images"], want_dir=True)
assert ANNOTATIONS, "ODIR-5K not attached: Add Input -> search 'Ocular Disease Recognition' (by Larxel / andrewmvd) -> Add."
assert IMAGE_DIR, "Could not find the 'Training Images' folder of ODIR-5K (raw photos are needed, not preprocessed ones)."
print("annotations:", ANNOTATIONS)
print("images     :", IMAGE_DIR, "|", len(os.listdir(IMAGE_DIR)), "files")

## 3. Per-eye labels from the diagnostic keywords

In [ ]:
raw = pd.read_excel(ANNOTATIONS) if ANNOTATIONS.endswith(".xlsx") else pd.read_csv(ANNOTATIONS)
patients = raw.drop_duplicates("ID")[["ID", "Left-Fundus", "Right-Fundus",
                                      "Left-Diagnostic Keywords", "Right-Diagnostic Keywords"]]

def terms(text):
    """ODIR keywords are separated by Chinese or Latin commas."""
    return [t.strip().lower() for t in re.split(r"[，,;]", str(text)) if t.strip()]

rows = []
for _, p in patients.iterrows():
    for side in ("Left", "Right"):
        rows.append({"patient_id": int(p["ID"]), "eye": side.lower(), "file": str(p[f"{side}-Fundus"]),
                     "terms": terms(p[f"{side}-Diagnostic Keywords"])})
eyes = pd.DataFrame(rows)
eyes["path"] = eyes["file"].map(lambda f: os.path.join(IMAGE_DIR, f))
eyes["exists"] = eyes["path"].map(os.path.exists)
eyes["unusable"] = eyes["terms"].map(lambda ts: any(u in t for t in ts for u in UNUSABLE))

def label(ts, spec):
    """1 = condition named for this eye, 0 = not named, None = ambiguous/excluded for this condition."""
    if any(a in t for t in ts for a in spec["ambiguous"] + spec["extra_exclusions"]):
        return None
    return int(any(spec["positive"] in t for t in ts))

for name, spec in CONDITIONS.items():
    eyes[name] = eyes["terms"].map(lambda ts, s=spec: label(ts, s))

print(f"patients: {len(patients)} | eye images listed: {len(eyes)} | image file missing: {(~eyes.exists).sum()} "
      f"| unusable photo: {eyes.unusable.sum()}")

## 4. True class counts (read this before trusting anything below)

In [ ]:
usable = eyes[eyes.exists & ~eyes.unusable]
counts = {}
for name in CONDITIONS:
    d = usable[usable[name].notna()]
    counts[name] = {
        "eyes": int(len(d)), "positive_eyes": int(d[name].sum()), "negative_eyes": int((d[name] == 0).sum()),
        "patients": int(d.patient_id.nunique()),
        "positive_patients": int(d[d[name] == 1].patient_id.nunique()),
        "excluded_ambiguous": int(usable[name].isna().sum()),
    }
count_table = pd.DataFrame(counts).T
print("TRUE CLASS COUNTS IN ODIR-5K (usable photos, per eye)")
print(count_table.to_string())
for name, c in counts.items():
    share = c["positive_eyes"] / max(c["eyes"], 1)
    expected_test = c["positive_eyes"] * 0.15
    print(f"\n{name}: {c['positive_eyes']} positive eyes ({share:.1%}) from {c['positive_patients']} patients; "
          f"about {expected_test:.0f} expected in the test split.")
    if c["positive_eyes"] * 0.70 < MIN_TRAIN_POSITIVES or expected_test < MIN_TEST_POSITIVES:
        print(f"  WARNING: positives are too few to trust a {name} model; results will be reported as indicative only.")
    else:
        print("  Positive class is small but large enough to train and test; intervals will still be wide.")

## 5. Patient-separated split

In [ ]:
# One patient-level split shared by both conditions: 70% train / 15% validation / 15% test.
per_patient = usable.groupby("patient_id").agg(
    g=("glaucoma", lambda s: int((s == 1).any())), c=("cataract", lambda s: int((s == 1).any()))).reset_index()
per_patient["strata"] = per_patient.g.astype(str) + per_patient.c.astype(str)

def split(ids, strata, size):
    try:
        return train_test_split(ids, test_size=size, random_state=SEED, stratify=strata)
    except ValueError:  # a stratum too small to split (only in tiny smoke data)
        return train_test_split(ids, test_size=size, random_state=SEED)

train_ids, rest_ids = split(per_patient.patient_id, per_patient.strata, 0.30)
rest = per_patient[per_patient.patient_id.isin(rest_ids)]
val_ids, test_ids = split(rest.patient_id, rest.strata, 0.50)
SPLIT_OF = {**{i: "train" for i in train_ids}, **{i: "val" for i in val_ids}, **{i: "test" for i in test_ids}}
usable = usable.assign(split=usable.patient_id.map(SPLIT_OF))

overlap = (set(train_ids) & set(val_ids)) | (set(train_ids) & set(test_ids)) | (set(val_ids) & set(test_ids))
assert not overlap, f"patients in more than one split: {sorted(overlap)[:5]}"
print(f"patients -> train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)} | overlap: none")
for name in CONDITIONS:
    d = usable[usable[name].notna()]
    print(name, {s: f"{int(g[name].sum())}+ / {len(g)}" for s, g in d.groupby("split")})

## 6. Load images with the app's preprocessing

In [ ]:
def load(path):
    """Exactly the app's preprocessing (backend/app/ml/preprocessing.py): RGB, nearest-neighbour resize to 224."""
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB").resize(IMG_SIZE, Image.NEAREST), dtype=np.uint8)

start = time.time()
paths = sorted(usable.path.unique())
with ThreadPoolExecutor(8) as pool:
    CACHE = dict(zip(paths, pool.map(load, paths)))
print(f"cached {len(CACHE)} images in {time.time() - start:.0f} s")

## 7. Model and data pipeline

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomContrast(0.1),
])

def dataset(frame, name, training):
    x = np.stack([CACHE[p] for p in frame.path])
    y = frame[name].astype("float32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
    def prep(img, lab):
        img = tf.cast(img, tf.float32)
        if training:
            img = augment(img, training=True)
        return img / 127.5 - 1.0, lab   # MobileNetV2 scaling, as in the app
    return ds.batch(BATCH).map(prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def build_model():
    base = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False,
                                             weights=None if SMOKE else "imagenet")
    base.trainable = False
    inputs = tf.keras.Input((*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model(inputs, outputs)
    return model, base

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=[tf.keras.metrics.AUC(name="auc")])

## 8. Evaluation helpers (same algorithms as the app's evaluation code)

In [ ]:
def select_threshold(y, p, target):
    """Same algorithm as backend/app/ml/thresholding.py: highest threshold with sensitivity >= target."""
    positives = np.sort(p[y == 1])
    candidates = np.unique(p)[::-1]
    caught = len(positives) - np.searchsorted(positives, candidates, side="left")
    eligible = np.flatnonzero(caught / len(positives) >= target)
    return float(candidates[eligible[0]])

def confusion(y, p, t):
    pred = p >= t
    tp = int(np.sum(pred & (y == 1))); fp = int(np.sum(pred & (y == 0)))
    tn = int(np.sum(~pred & (y == 0))); fn = int(np.sum(~pred & (y == 1)))
    div = lambda a, b: float(a / b) if b else 0.0
    sens, spec, prec = div(tp, tp + fn), div(tn, tn + fp), div(tp, tp + fp)
    return {"threshold": float(t), "sensitivity": sens, "specificity": spec, "precision": prec,
            "f1": div(2 * prec * sens, prec + sens), "accuracy": div(tp + tn, len(y)),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def patient_bootstrap(y, p, groups, t, n=BOOTSTRAP):
    """95% interval resampling *patients* (both eyes stay together), so correlated eyes are not double-counted."""
    rng = np.random.default_rng(SEED)
    ids = np.unique(groups)
    index = {g: np.flatnonzero(groups == g) for g in ids}
    out = {"sensitivity": [], "specificity": [], "precision": []}
    for _ in range(n):
        rows = np.concatenate([index[g] for g in rng.choice(ids, size=len(ids), replace=True)])
        if y[rows].min() == y[rows].max():
            continue
        m = confusion(y[rows], p[rows], t)
        for k in out:
            out[k].append(m[k])
    point = confusion(y, p, t)
    return {k: {"estimate": point[k], "low": float(np.quantile(v, 0.025)), "high": float(np.quantile(v, 0.975))}
            for k, v in out.items() if v}

def auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(y, p)) if y.min() != y.max() else float("nan")

def tflite_predict(tflite_bytes, frame):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    scores = []
    for path in frame.path:
        x = (CACHE[path].astype(np.float32) / 127.5 - 1.0)[None]
        interp.set_tensor(inp["index"], x)
        interp.invoke()
        scores.append(float(interp.get_tensor(out["index"]).reshape(-1)[0]))
    return np.array(scores), [int(d) for d in inp["shape"]]

def to_tflite(model):
    try:
        return tf.lite.TFLiteConverter.from_keras_model(model).convert()
    except Exception as exc:  # some Keras 3 versions need a SavedModel round trip
        print("from_keras_model failed, exporting a SavedModel first:", type(exc).__name__)
        export_dir = os.path.join(OUT_ROOT, "tmp_saved_model")
        model.export(export_dir)
        return tf.lite.TFLiteConverter.from_saved_model(export_dir).convert()

## 9. Train, export to TFLite, choose threshold on validation, score test once

In [ ]:
RESULTS = {}
for name, spec in CONDITIONS.items():
    print("=" * 70, f"\n{name.upper()}")
    d = usable[usable[name].notna()].copy()
    d[name] = d[name].astype(int)
    train, val, test = (d[d.split == s] for s in ("train", "val", "test"))
    n_pos, n_neg = int(train[name].sum()), int((train[name] == 0).sum())
    class_weight = {0: (n_pos + n_neg) / (2 * n_neg), 1: (n_pos + n_neg) / (2 * max(n_pos, 1))}

    tf.keras.utils.set_random_seed(SEED)
    model, base = build_model()
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    compile_model(model, 1e-4)
    h1 = model.fit(dataset(train, name, True), validation_data=dataset(val, name, False),
                   epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=[stop], verbose=2)
    # Fine-tune the top of the backbone; BatchNorm layers stay frozen.
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, 1e-5)
    h2 = model.fit(dataset(train, name, True), validation_data=dataset(val, name, False),
                   epochs=EPOCHS_FINE, class_weight=class_weight, callbacks=[stop], verbose=2)

    tflite = to_tflite(model)
    keras_val = model.predict(dataset(val, name, False), verbose=0).ravel()
    p_val, input_shape = tflite_predict(tflite, val)
    p_test, _ = tflite_predict(tflite, test)
    y_val, y_test = val[name].to_numpy(), test[name].to_numpy()

    # Threshold from the VALIDATION split only; the test split is scored once, below.
    threshold = select_threshold(y_val, p_val, TARGET_SENSITIVITY)
    RESULTS[name] = {
        "tflite": tflite, "input_shape": input_shape, "threshold": threshold,
        "epochs_run": len(h1.history["loss"]) + len(h2.history["loss"]),
        "max_keras_tflite_diff": float(np.abs(keras_val - p_val).max()),
        "counts": {s: {"eyes": int(len(f)), "positives": int(f[name].sum()), "patients": int(f.patient_id.nunique())}
                   for s, f in (("train", train), ("val", val), ("test", test))},
        "test_at_threshold": confusion(y_test, p_test, threshold),
        "test_at_0_5": confusion(y_test, p_test, 0.5),
        "val_at_threshold": confusion(y_val, p_val, threshold),
        "auc_test": auc(y_test, p_test),
        "ci": patient_bootstrap(y_test, p_test, test.patient_id.to_numpy(), threshold),
        "scores_in_range": bool((p_test >= 0).all() and (p_test <= 1).all()),
        "splits_disjoint": not (set(train.patient_id) & set(test.patient_id) or set(val.patient_id) & set(test.patient_id)
                                or set(train.patient_id) & set(val.patient_id)),
    }
    r = RESULTS[name]
    print(f"threshold (from validation): {threshold:.3f} | test AUC {r['auc_test']:.3f}")
    print("test @ threshold:", {k: round(v, 3) if isinstance(v, float) else v for k, v in r["test_at_threshold"].items()})

## 10. Write model.tflite + model_card.json for the app

In [ ]:
os.makedirs(os.path.join(OUT_ROOT, "models"), exist_ok=True)
summary = {}
for name, r in RESULTS.items():
    folder = os.path.join(OUT_ROOT, "models", name)
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, "model.tflite"), "wb") as f:
        f.write(r["tflite"])
    sha = hashlib.sha256(r["tflite"]).hexdigest()
    m, c = r["test_at_threshold"], r["counts"]
    trustworthy = c["test"]["positives"] >= MIN_TEST_POSITIVES and c["train"]["positives"] >= MIN_TRAIN_POSITIVES
    meets = m["sensitivity"] >= TARGET_SENSITIVITY and m["specificity"] >= TARGET_SPECIFICITY
    status = (f"measured on a patient-separated held-out test split ({c['test']['eyes']} eyes, "
              f"{c['test']['positives']} positive, {c['test']['patients']} patients); limited evidence, not clinically validated"
              + ("" if trustworthy else "; TOO FEW POSITIVE CASES TO TRUST, indicative only")
              + ("; meets the 85%/80% targets" if meets else "; below the 85% sensitivity / 80% specificity targets"))
    card = {
        "condition": name, "display_name": CONDITIONS[name]["display_name"],
        "version": f"mobilenetv2-{name}-binary-odir5k-0.1", "file": "model.tflite", "sha256": sha, "task": "binary",
        "labels": {"0": f"No {name} named for this eye", "1": f"{CONDITIONS[name]['display_name']} named in the eye's diagnostic keywords"},
        "input": {"shape": r["input_shape"], "dtype": "float32", "color": "RGB",
                  "resize": "nearest-neighbour to 224x224 (matches the app's ImagePreprocessor)",
                  "normalisation": "MobileNetV2 preprocess_input: pixel / 127.5 - 1, range [-1, 1]"},
        "output": {"shape": [1, 1], "meaning": f"sigmoid score for {name} present, in [0, 1]"},
        "threshold": r["threshold"],
        "threshold_source": "selected on the patient-separated validation split: highest threshold with sensitivity >= 85%; the test split was not used for any choice",
        "architecture": "MobileNetV2 (ImageNet weights) -> GlobalAveragePooling2D -> Dropout(0.3) -> Dense(64, relu) -> Dropout(0.2) -> Dense(1, sigmoid); head trained with Adam 1e-4, then top 30 backbone layers fine-tuned at 1e-5 (BatchNorm frozen); class weights; early stopping on validation loss; SEED 42",
        "training": {
            "dataset": "Kaggle andrewmvd/ocular-disease-recognition-odir5k (ODIR-5K training set, per-eye diagnostic keywords)",
            "label_rule": f"positive if '{CONDITIONS[name]['positive']}' appears in that eye's keywords; excluded: {CONDITIONS[name]['ambiguous'] + CONDITIONS[name]['extra_exclusions']} and unusable photos {UNUSABLE}",
            "split": "by patient, 70/15/15, stratified on whether a patient has glaucoma and/or cataract; no patient in more than one split",
            "counts": c, "epochs_run": r["epochs_run"],
            "notebook": "notebooks/odir_glaucoma_cataract.ipynb",
            "export": f"float32 TFLite; max |Keras - TFLite| score difference on validation {r['max_keras_tflite_diff']:.2e}",
        },
        "metrics": {
            "sensitivity": m["sensitivity"], "specificity": m["specificity"], "precision": m["precision"],
            "f1": m["f1"], "accuracy": m["accuracy"], "auc": r["auc_test"], "test_images": c["test"]["eyes"],
            "test_positives": c["test"]["positives"], "test_patients": c["test"]["patients"],
            "confidence_intervals": r["ci"], "at_default_threshold_0_5": r["test_at_0_5"],
            "source": "notebooks/odir_glaucoma_cataract.ipynb (test split scored once with the exported TFLite file)",
        },
        "metrics_status": status,
        "limitations": [
            "Screening support only. Not a diagnosis and not clinically validated.",
            "Binary result only: ODIR-5K has no severity grades, so no severity band is given.",
            "Labels come from free-text diagnostic keywords per eye in ODIR-5K and may be noisy.",
            f"Small positive class: {c['test']['positives']} positive test eyes, so the intervals are wide.",
            "Single public dataset collected in one country; no external validation; no smartphone-lens images.",
        ],
    }
    with open(os.path.join(folder, "model_card.json"), "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2)
    summary[name] = {"metrics": card["metrics"], "status": status, "threshold": r["threshold"],
                     "trustworthy": trustworthy, "meets_targets": meets}

summary["class_counts"] = counts
with open(os.path.join(OUT_ROOT, "odir_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
with zipfile.ZipFile(os.path.join(OUT_ROOT, "odir_models.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    for name in RESULTS:
        for fn in ("model.tflite", "model_card.json"):
            z.write(os.path.join(OUT_ROOT, "models", name, fn), f"{name}/{fn}")
    z.write(os.path.join(OUT_ROOT, "odir_metrics.json"), "odir_metrics.json")
print("wrote", os.path.join(OUT_ROOT, "odir_models.zip"))

## 11. CHECKS (screenshot this)

In [ ]:
print("CHECKS")
all_ok = True
for name, r in RESULTS.items():
    folder = os.path.join(OUT_ROOT, "models", name)
    card = json.load(open(os.path.join(folder, "model_card.json"), encoding="utf-8"))
    file_sha = hashlib.sha256(open(os.path.join(folder, "model.tflite"), "rb").read()).hexdigest()
    checks = {
        "patients never in two splits": r["splits_disjoint"],
        "threshold chosen on validation only": card["threshold"] == r["threshold"] and 0 < r["threshold"] < 1,
        "card SHA-256 matches the exported file": card["sha256"] == file_sha,
        "TFLite input is [1, 224, 224, 3]": r["input_shape"] == [1, 224, 224, 3] == card["input"]["shape"],
        "TFLite agrees with Keras (max diff < 1e-3)": r["max_keras_tflite_diff"] < 1e-3,
        "test scores are probabilities in [0, 1]": r["scores_in_range"],
        "card has every field the app needs": all(k in card for k in ("condition", "version", "sha256", "input", "labels", "threshold", "metrics", "limitations")),
    }
    for label_, ok in checks.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}: {label_}")
        all_ok &= ok
    s = summary[name]
    reliability = "OK to report (limited)" if s["trustworthy"] else "TOO FEW POSITIVES: indicative only"
    target = "meets targets" if s["meets_targets"] else "BELOW targets"
    m = s["metrics"]
    print(f"  {name}: sensitivity {m['sensitivity']:.3f}, specificity {m['specificity']:.3f}, AUC {m['auc']:.3f} "
          f"on {m['test_images']} test eyes ({m['test_positives']} positive) -> {target}; reliability: {reliability}")
print("\nOVERALL:", "ALL INTEGRITY CHECKS PASSED" if all_ok else "SOME CHECKS FAILED: do not use these models")
print("Weak results are still valid results: report them as measured, labelled limited and not clinically validated.")